### ============================================================
### DATA COLLECTION NOTEBOOK — DO NOT RE-RUN
### Outputs are already saved to /data/. Re-running requires API connection and will overwrite archived raw data.
### ============================================================

# 1
In this notebook, I connect to the API and collect the debates from 2010-2026 of the Oireachtas archives. Then, using keywords and high-profile homes, I collect the speeches on mother and baby homes. 

In [ ]:
import requests
import csv
from time import sleep
import xml.etree.ElementTree as ET

In [ ]:
# Parameters
years = range(2010, 2026)  # 2010 through 2025
limit = 100  # max results per page
base_url = "https://api.oireachtas.ie/v1/debates"

In [ ]:
# Fetch data

for year in years:
    date_start = f"{year}-01-01"
    date_end = f"{year}-12-31"
    output_csv = f"final_debates_{year}_metadata.csv"

    print(f"\nFetching debates for {year}...")

    seen_uris = set()
    metadata_rows = []

    skip = 0
    while True:
        params = {
            "date_start": date_start,
            "date_end": date_end,
            "limit": limit,
            "skip": skip
        }
        r = requests.get(base_url, params=params) # HTTP get request sent, API responds with JSON  
        if r.status_code != 200:
            print(f"Error fetching data: {r.status_code}")
            break

        data = r.json() # convert into a python dictionary
        results = data.get("results", [])

        if not results:
            break

        new_count = 0
        for item in results:
            debate = item.get("debateRecord", {})
            uri = debate.get("uri")
            if uri and uri not in seen_uris:
                seen_uris.add(uri)
                date = debate.get("date")
                house = debate.get("house", {}).get("houseCode")
                debate_type = debate.get("debateType")
                xml_uri = debate.get("formats", {}).get("xml", {}).get("uri")
                metadata_rows.append([date, house, debate_type, uri, xml_uri])
                new_count += 1

        skip += limit
        print(f"  Fetched page: {skip//limit}, new unique debates: {new_count}, total unique so far: {len(seen_uris)}")

        if new_count == 0:
            break

        sleep(0.2)  # polite pause

    # save CSV
    with open(output_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["date", "house", "debateType", "debate_uri", "xml_uri"])
        writer.writerows(metadata_rows)

    print(f"Metadata for {year} saved to {output_csv}, total unique debates: {len(seen_uris)}")

In [ ]:
# Define keywords
GENERIC_KEYWORDS = [
    "mother and baby home",
    "mother and baby homes",
    "mother-and-baby home",
    "mother-and-baby homes",
    "mother & baby home",
    "mother & baby homes"
]

HIGH_PROFILE_HOMES = [
    "tuam",
    "bessborough",
    "bethany home"
]


In [ ]:
# parameters
years = list(range(2010, 2026))
ns = {'akn': 'http://docs.oasis-open.org/legaldocml/ns/akn/3.0/CSD13'}

# fetch datas
for year in years:
    
    metadata_file = f"raw_data/final_debates_{year}_metadata.csv"
    output_file = f"raw_data/final_speeches_{year}_mother_and_baby_homes_speeches.csv"

    # Read metadata CSV
    with open(metadata_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        metadata_rows = list(reader)

    # Open CSV for writing
    with open(output_file, "w", encoding="utf-8", newline="") as f_out:
        fieldnames = ["date", "house", "debate_uri", "speech_number", "speaker", "member_name", "text"]
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()

        matches = 0
        total = len(metadata_rows)

        for i, row in enumerate(metadata_rows, start=1):
            xml_url = row['xml_uri']
            try:
                r = requests.get(xml_url)
                if r.status_code == 200:
                    root = ET.fromstring(r.content)
                    
                    # Extract speeches
                    for j, s in enumerate(root.findall('.//akn:speech', ns), start=1):
                        # Get speaker if available
                        speaker_tag = s.find('akn:speaker', ns)
                        if speaker_tag is not None:
                            speaker = "".join(speaker_tag.itertext()).strip()
                        else:
                            # fallback: first line of text
                            full_text = "".join(s.itertext()).strip()
                            speaker = full_text.splitlines()[0] if full_text else "Unknown"

                        # MEMBER NAME extraction (<from>)
                        from_tag = s.find('akn:from', ns)
                        if from_tag is not None:
                            member_name = "".join(from_tag.itertext()).strip()
                        else:
                            member_name = ""

                        text = "".join(s.itertext()).strip().lower()
                        
                        # Matching logic
                        if any(keyword in text for keyword in GENERIC_KEYWORDS) or \
                           any(home in text and "mother" in text and "baby" in text for home in HIGH_PROFILE_HOMES): #AND to make sure its on the topic
                            writer.writerow({
                                "date": row["date"],
                                "house": row["house"],
                                "debate_uri": row["debate_uri"],
                                "speech_number": j,
                                "speaker": speaker,
                                "member_name": member_name,
                                "text": text
                            })
                            matches += 1


                else:
                    print(f"Failed to fetch {xml_url}, status {r.status_code}")
            except ET.ParseError:
                print(f"Failed to parse XML from {xml_url}")

            # Progress display
            if i % 10 == 0 or i == total:
                print(f"[{year}] Processed {i}/{total} debates, matches found: {matches}")
            sleep(0.1)  # polite pause

    print(f"[{year}] Step 2 complete! Total speeches containing 'mother and baby homes': {matches}")